## Projet Santé Mentale des Adolescents

In [664]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.2 s3fs==2026.3.0 -q

Note: you may need to restart the kernel to use updated packages.


## Importation des packages nécessaires

In [665]:
import pandas as pd
import os
import openpyxl
from openpyxl import *
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import *
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import Workbook   
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import load_workbook 
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Font, Border, Side
from openpyxl.styles import Alignment
from openpyxl.chart.label import DataLabelList                                                                                                                                                      
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import quote_sheetname
from openpyxl.utils.cell import coordinate_from_string, column_index_from_string
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.styles import Alignment, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
import pandas as pd
from PIL import Image

print(openpyxl.__version__)


3.1.3


### Importation des données - Santé Mentale

Après l'importation, on inspecte les types de données présents.

In [666]:
df = pd.read_csv('https://minio.lab.sspcloud.fr/nerojeni10/DATA_PROJET_SMA/Teen_Mental_Health_Dataset.csv')

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   str    
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   str    
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   str    
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
dtypes: float64(5), int64(5), str(3)
memory usage: 140.4 KB


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


### Inspecter la présence des valeurs manquantes

In [667]:
df.isnull().sum()

age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64

***Il n'y a aucune valeur manquante dans ce jeux de données***

## Remplissage du fichier

Pour remplir le fichier, on allons créer plusieurs feuilles composées des données nécessaires à la création des indicateurs

In [668]:
path_file = "../template/Projet_ODD_SIVARAJAH.xlsx"

# Recréer un fichier propre sans feuille parasite
try:
    wb = load_workbook(path_file)  # noqa: F405
except Exception:
    wb = Workbook()


# Créer un vrai fichier Excel vide si inexistant
if not os.path.exists(path_file):
    wb = Workbook()
    wb.save(path_file)


# Ajouter la feuille DATA
with pd.ExcelWriter(path_file, mode="a", if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name='DATA', index=False)



print("Feuilles présentes :", load_workbook(path_file).sheetnames)

Feuilles présentes : ['Sheet', 'DATA']


## Création de la feuille CALC

Sur cette feuille, on  aura les valeurs distinctes pour chaque variable, afin de réaliser des groupes plus tard et de réaliser des agrégations dessus.

In [669]:
# # Chargement du fichier en mémoire
# wb = load_workbook(path_file)

# # Créer la feuille CALC si elle n'existe pas
# if "CALC" not in wb.sheetnames:
#     ws = wb.create_sheet("CALC")
# else:
#     ws = wb["CALC"]

### Création des variables distinctes

In [670]:
# from openpyxl.utils import FORMULAE
# "UNIQUE" in FORMULAE

# # ws["A1"]="Genres distincts"
# formula = "=_xlfn.UNIQUE(DATA!B2:B)"
# ws["A1"]=ArrayFormula("A1:A", formula)

# # # ws["A3"]="Ages distincts"
# # # ws["A3"]= "=_xlfn.UNIQUE(DATA!A2:A)"

# # # ws["A5"]="Plateformes distincts"
# # # ws["A5"]= "=_xlfn.UNIQUE(DATA!D2:D)"

# # # ws["A7"]="Social Interactions"
# # # ws["A7"]= "=_xlfn.UNIQUE(DATA!I2:I)"


# # wb.save(path_file)


## Création des indicateurs

In [671]:
# Chargement du fichier en mémoire
wb = load_workbook(path_file)


# Supprimer la feuille vide par défaut si elle existe
if "Sheet" in wb.sheetnames:
    del wb["Sheet"]

wb.save(path_file)

# Créer la feuille Indicateurs si elle n'existe pas
if "Indicateurs" not in wb.sheetnames:
    ws = wb.create_sheet("Indicateurs")
else:
    ws = wb["Indicateurs"]

# Ajout des formules
# 1. Nombre de filles dépressives
ws['A1'] = "Nombre de filles dépressives"
ws['B1'] = '=COUNTIFS(DATA!M:M,1,DATA!B:B,"female")'

# 2. Nombre de garçons dépressifs
ws['A2'] = "Nombre de garçons dépressifs"
ws['B2'] = '=COUNTIFS(DATA!M:M,1,DATA!B:B,"male")'

# 3. Niveau d'addiction moyen chez les filles
ws['A3'] = "Niveau d'addiction moyen chez les filles"
ws['B3'] = '=AVERAGEIF(DATA!B:B,"female",DATA!L:L)'

# 4. Niveau d'addiction moyen chez les garçons
ws['A4'] = "Niveau d'addiction moyen chez les garçons"
ws['B4'] = '=AVERAGEIF(DATA!B:B,"male",DATA!L:L)'


# Création d'une nouvelle feuille, TCD (Tableau croisé dynamique)

Sur cette feuille apparaîtrant les indicateurs qui sont groupés selon différents critères comme l'âge ou le genre. 

Pour plus de simplicité, la réalisation de ces groupes et des agrégations nécessaires j'utilise la bibliothèque pandas et les résultats sont par la suite transcris dans les feuilles. 

Afin d'automatiser l'écriture des données, et d'éviter le chevauchement des résultats une fonction est crée pour CALCuler automatiquement la cellule dans la  quelle on commencera à écrire les données.

In [672]:
def write_table(ws, df, start_row, title=None, space=3, padding=2):
    """
    Écrit un DataFrame dans une feuille Excel OpenPyXL à partir d'une ligne donnée.

    La fonction ajoute éventuellement un titre, écrit les en-têtes de colonnes
    puis les données du DataFrame. Elle retourne ensuite la première ligne
    disponible pour écrire un nouveau tableau en laissant un nombre de lignes
    vides configurable.

    Args:
        ws: Feuille OpenPyXL cible.
        df: DataFrame à écrire.
        start_row (int): Ligne de départ.
        title (str, optional): Titre du tableau.
        space (int, optional): Nombre de lignes vides à laisser après le tableau.

    Returns:
        int: Numéro de la prochaine ligne disponible.

    Examples:
    >>> start_row = 1
    >>> start_row = write_table(ws, tcd1, start_row,
    ...                         "Dépression selon l'âge")
    >>> start_row = write_table(ws, tcd2, start_row,
    ...                         "Addiction moyenne selon l'âge et le genre")
    """

    # En-têtes
    if title:
        ws.cell(row=start_row, column=1, value=title)
        start_row += 1

    # En-têtes + ajustement largeur colonnes
    for col_idx, header in enumerate(df.columns, start=1):
        ws.cell(row=start_row, column=col_idx, value=header)

        col_letter = get_column_letter(col_idx)
        width = len(str(header)) + padding

        # On conserve la plus grande largeur si la colonne existe déjà
        current_width = ws.column_dimensions[col_letter].width
        if current_width is None or width > current_width:
            ws.column_dimensions[col_letter].width = width

    # Données
    for i, row in df.iterrows():
        for col_idx, value in enumerate(row, start=1):
            ws.cell(
                row=start_row + i + 1,
                column=col_idx,
                value=value
            )

    # Ligne de départ du tableau suivant
    return start_row + len(df) + space + 1

### Création de la feuille TCD si elle n'existe pas

In [673]:
if "TCD" not in wb.sheetnames:
    ws_tcd = wb.create_sheet("TCD")
else:
    ws_tcd = wb["TCD"]


### Création des indicateurs et écriture des données avec la fonction créée

### Création d'une matrice de corrélation

Pour étudier les liens entre les différents variables de ce jeux de données

In [674]:
# Encoder les variables catégorielles en numérique
df["gender_num"] = df["gender"].map({"male": 0, "female": 1})
df["social_num"] = df["social_interaction_level"].map({"low": 0, "medium": 1, "high": 2})

# Sélectionner les colonnes numériques
cols_corr = ["age", "sleep_hours", "daily_social_media_hours",
             "academic_performance", "physical_activity",
             "social_num", "stress_level", "anxiety_level",
             "addiction_level", "depression_label"]

# Matrice de corrélation
corr = df[cols_corr].corr().round(2)

corr_reset = corr.reset_index()
corr_reset.columns = ["Variable"] + cols_corr

# Création d'une nouvelle feuille pour réaliser le tableau de corrélation
wb = load_workbook(path_file)

if "Correlations" not in wb.sheetnames:
    ws_corr = wb.create_sheet("Correlations")
else:
    ws_corr = wb["Correlations"]

write_table(
    ws_corr,
    corr_reset,
    start_row=1,
    title="Matrice de corrélation",
    space=0
)

# Sauvegarde du fichier
wb.save(path_file)
wb.close()
print("Feuilles presentes :", load_workbook(path_file).sheetnames)

Feuilles presentes : ['DATA', 'Correlations']


In [675]:

cols_to_calculate = ['age', 'gender', 'platform_usage', 'social_interaction_level']
len_dict ={}
for col in cols_to_calculate:
    len_dict[f"len_{col}"] = len(df[col].unique())+1 
print(f'{len_dict}')



{'len_age': 8, 'len_gender': 3, 'len_platform_usage': 4, 'len_social_interaction_level': 4}


In [676]:
from openpyxl import load_workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.worksheet.table import Table, TableStyleInfo

wb = load_workbook(path_file)

if "CALC" not in wb.sheetnames:
    ws_calc = wb.create_sheet("CALC")
else:
    ws_calc = wb["CALC"]

style = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)

# Genres - colonne A
formula = "=_xlfn.UNIQUE(DATA!B:B)"
ws_calc['A1'] = ArrayFormula(
    f"A1:A{len_dict['len_gender']}",
    formula
)
table_gender = Table(displayName="tblGenres", ref=f"A1:A{len_dict['len_gender']}")
table_gender.tableStyleInfo = style
table_gender.hasHeader = False
ws_calc.add_table(table_gender)

# Ages - colonne C
formula = "=_xlfn.UNIQUE(DATA!A:A)"
ws_calc['C1'] = ArrayFormula(
    f"C1:C{len_dict['len_age']}",
    formula
)
table_age = Table(displayName="tblAges", ref=f"C1:C{len_dict['len_age']}")
table_age.tableStyleInfo = style
table_age.hasHeader = False
ws_calc.add_table(table_age)

# Plateformes - colonne E
formula = "=_xlfn.UNIQUE(DATA!D:D)"
ws_calc['E1'] = ArrayFormula(
    f"E1:E{len_dict['len_platform_usage']}",
    formula
)
table_platform = Table(displayName="tblPlateformes", ref=f"E1:E{len_dict['len_platform_usage']}")
table_platform.tableStyleInfo = style
table_platform.hasHeader = False
ws_calc.add_table(table_platform)

# Interactions sociales - colonne G
formula = "=_xlfn.UNIQUE(DATA!I:I)"
ws_calc['G1'] = ArrayFormula(
    f"G1:G{len_dict['len_social_interaction_level']}",
    formula
)
table_interaction = Table(displayName="tblInteractions", ref=f"G1:G{len_dict['len_social_interaction_level']}")
table_interaction.tableStyleInfo = style
table_interaction.hasHeader = False
ws_calc.add_table(table_interaction)

wb.save(path_file)
wb.close()
print("Feuilles presentes :", load_workbook(path_file).sheetnames)

Feuilles presentes : ['DATA', 'Correlations', 'CALC']


/opt/python/lib/python3.13/site-packages/openpyxl/worksheet/_writer.py:274: UserWarning: File may not be readable: column headings must be strings.
  warn("File may not be readable: column headings must be strings.")


# Création du dashboard

## Création des filtres

Maintenant qu'on a les indicateurs uniques, on peut les utiliser pour la création de filtres.

In [677]:
# from openpyxl import load_workbook
# from openpyxl.styles import Alignment, PatternFill, Font, Border, Side
# from openpyxl.worksheet.datavalidation import DataValidation

# wb = load_workbook(path_file)

# # 1. Préparation de la feuille TDB_1
# if "TDB_1" in wb.sheetnames:
#     del wb["TDB_1"]
# TDB1_sheet = wb.create_sheet("TDB_1")
# TDB1_sheet.sheet_view.showGridLines = False 

# # 2. Titre encadré (B2:P3)
# TDB1_sheet.merge_cells('B2:P3')
# titre = TDB1_sheet['B2']
# titre.value = "Impact des réseaux sociaux sur la santé mentale"
# titre.font = Font(size=22, bold=True, color="333333")
# titre.alignment = Alignment(horizontal='center', vertical='center')

# # Application de la bordure sur la fusion
# medium_border = Border(left=Side(style='medium'), right=Side(style='medium'), 
#                        top=Side(style='medium'), bottom=Side(style='medium'))

# for row in TDB1_sheet['B2:P3']:
#     for cell in row:
#         cell.border = medium_border

# wb.save(path_file)
# wb.close()

In [678]:
from openpyxl.styles import Alignment, PatternFill, Font, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation

def add_filter(worksheet, start_col, filter_row, title_text, 
               data_source_col, len_data, helper_col, default_value='Tous', 
               title_color='FF008080', value_color='FFB3E5E0'):
    """
    Crée un filtre horizontal avec titre et valeur côte à côte.
    
    Le filtre génère une colonne cachée (helper_col) pour inclure l'option "Tous"
    sans modifier la feuille CALC d'origine. Les titres et valeurs ont des couleurs différentes.
    
    Parameters
    ----------
    worksheet : openpyxl.worksheet.worksheet.Worksheet
        La feuille de travail où ajouter le filtre.
    start_col : str
        Colonne de départ pour le titre (ex: 'C'). 
        La valeur sera dans la colonne suivante (ex: 'D').
    filter_row : int
        La ligne où placer le filtre (ex: 3).
    title_text : str
        Le texte du titre du filtre (ex: 'Âge', 'Genre').
    data_source_col : str
        La colonne source dans CALC (ex: 'C', 'A', 'E').
    len_data : int
        Le nombre de valeurs uniques (ex: len_dict['len_age']).
    helper_col : str
        La colonne cachée pour stocker les données (ex: 'AA', 'AB').
    default_value : str, optional
        La valeur par défaut affichée. Par défaut, 'Tous'.
    title_color : str, optional
        Couleur hexadécimale du titre (ex: 'FF008080'). Par défaut, teal foncé.
    value_color : str, optional
        Couleur hexadécimale de la valeur (ex: 'FFB3E5E0'). Par défaut, teal clair.
    
    Returns
    -------
    None
        Modifie la feuille en place.
    
    Examples
    --------
    >>> add_filter(ws, 'C', 3, 'Âge', 'C', 8, 'AB')
    # Crée un filtre "Âge" avec titre teal foncé et valeur teal clair
    
    Notes
    -----
    - Le titre est placé dans start_col, la valeur dans la colonne suivante.
    - L'option "Tous" est automatiquement ajoutée sans modifier CALC.
    - Les colonnes helper sont masquées du tableau de bord.
    """
    
    # Colonne de la valeur (juste après le titre)
    value_col = chr(ord(start_col) + 1)
    
    # Remplissage pour titre et valeur
    title_fill = PatternFill(start_color=title_color, end_color=title_color, fill_type='solid')
    value_fill = PatternFill(start_color=value_color, end_color=value_color, fill_type='solid')
    
    alignment = Alignment(horizontal='center', vertical='center')
    border = Border(
        left=Side(style='thin'), right=Side(style='thin'),
        top=Side(style='thin'), bottom=Side(style='thin')
    )
    title_font = Font(bold=True, color='FFFFFF', size=10)
    value_font = Font(bold=True, color='000000', size=10)  # Texte noir pour le clair
    
    # ========== TITRE DU FILTRE ==========
    title_cell = worksheet[f'{start_col}{filter_row}']
    title_cell.value = title_text
    title_cell.alignment = alignment
    title_cell.fill = title_fill
    title_cell.border = border
    title_cell.font = title_font
    worksheet.column_dimensions[start_col].width = 12
    
    # ========== CELLULE DE VALEUR (couleur plus claire) ==========
    value_cell = worksheet[f'{value_col}{filter_row}']
    value_cell.value = default_value
    value_cell.alignment = alignment
    value_cell.fill = value_fill
    value_cell.border = border
    value_cell.font = value_font
    worksheet.column_dimensions[value_col].width = 12
    
    # ========== COLONNE HELPER (CACHÉE) ==========
    worksheet[f'{helper_col}1'] = 'Tous'
    
    for i in range(2, len_data + 1):
        worksheet[f'{helper_col}{i}'] = f'=CALC!{data_source_col}{i}'
    
    worksheet.column_dimensions[helper_col].hidden = True
    
    # ========== VALIDATION DE DONNÉES ==========
    formula = f"=${helper_col}$1:${helper_col}${len_data}"
    
    dv = DataValidation(type='list', formula1=formula, allow_blank=False)
    dv.error = 'Sélectionnez une valeur valide'
    dv.errorTitle = 'Entrée invalide'
    dv.prompt = f'Sélectionnez un {title_text.lower()}'
    dv.promptTitle = 'Filtrer'
    
    worksheet.add_data_validation(dv)
    dv.add(f'{value_col}{filter_row}')
    
    print(f"✅ Filtre '{title_text}' créé en {start_col}{filter_row}:{value_col}{filter_row}")

# ============================================================================
# PAGE 1 - TITRE BLEU + FILTRES AVEC DISTINCTION COULEUR
# ============================================================================

print("\n📊 Création de TDB1 avec titre bleu et filtres contrastés...\n")

# Nettoyer si existe déjà
if "TDB1" in wb.sheetnames:
    del wb["TDB1"]
TDB1 = wb.create_sheet("TDB1", 0)
TDB1.sheet_view.showGridLines = False

# ========== TITRE PRINCIPAL EN BLEU FONCÉ ==========
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type='solid')  # Bleu foncé
title_font = Font(name='Calibri', size=14, bold=True, color='FFFFFF')
title_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

TDB1.merge_cells('A1:O1')
TDB1['A1'] = 'Impact des réseaux sociaux sur la santé mentale'
TDB1['A1'].fill = title_fill
TDB1['A1'].font = title_font
TDB1['A1'].alignment = title_alignment
TDB1.row_dimensions[1].height = 30

# ========== LIGNE DE FILTRES ==========
TDB1['A3'] = 'Filtres :'
TDB1['A3'].font = Font(size=10, bold=True)
TDB1['A3'].alignment = Alignment(horizontal='left', vertical='center')

# FILTRES HORIZONTAUX
# Format: add_filter(ws, start_col_titre, row, titre, col_CALC, len_data, col_helper, 
#                    default, title_color, value_color)

# Teal foncé pour titre (#008080), Teal clair pour valeur (#B3E5E0)
add_filter(TDB1, 'C', 3, 'Âge', 'C', len_dict['len_age'], 'AA', 
           title_color='FF008080', value_color='FFB3E5E0')

add_filter(TDB1, 'F', 3, 'Genre', 'A', len_dict['len_gender'], 'AB',
           title_color='FF008080', value_color='FFB3E5E0')

add_filter(TDB1, 'I', 3, 'Plateforme', 'E', len_dict['len_platform_usage'], 'AC',
           title_color='FF008080', value_color='FFB3E5E0')

add_filter(TDB1, 'L', 3, 'Interaction', 'G', len_dict['len_social_interaction_level'], 'AD',
           title_color='FF008080', value_color='FFB3E5E0')

# Définir la hauteur de la ligne des filtres
TDB1.row_dimensions[3].height = 25

wb.save(path_file)
print("\n✅ TDB1 créée avec succès !")
print(f"📋 Feuilles présentes : {wb.sheetnames}\n")
wb.close()


📊 Création de TDB1 avec titre bleu et filtres contrastés...

✅ Filtre 'Âge' créé en C3:D3
✅ Filtre 'Genre' créé en F3:G3
✅ Filtre 'Plateforme' créé en I3:J3
✅ Filtre 'Interaction' créé en L3:M3



✅ TDB1 créée avec succès !
📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC']



# Création des tableaux groupés

Ces tableaux permettront de créer les graphiques reposant sur plusieurs critères par exemple.

In [679]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

wb = load_workbook(path_file)

# ========== STYLES ==========
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")  # Bleu foncé
title_font = Font(bold=True, size=12, color="FFFFFF")
header_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")  # Teal foncé
header_font = Font(color="FFFFFF", bold=True, size=10)
row_header_fill = PatternFill(start_color="FFB3E5E0", end_color="FFB3E5E0", fill_type="solid")  # Teal clair
row_header_font = Font(bold=True, color="000000", size=10)
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center")

# ========== CRÉATION DE LA FEUILLE ==========
if "TCD" in wb.sheetnames:
    del wb["TCD"]
ws_tcd = wb.create_sheet("TCD")

# =================================================================
# TABLEAU TCD DYNAMIQUE : Dépression par Genre et Âge
# (N'affiche que les lignes/colonnes correspondant aux filtres)
# =================================================================

# --- TITRE ---
ws_tcd['A1'] = "Dépression par Genre et Âge"
ws_tcd['A1'].font = title_font
ws_tcd['A1'].fill = title_fill
ws_tcd['A1'].alignment = center_align
ws_tcd.merge_cells('A1:J1')
ws_tcd.row_dimensions[1].height = 25

# --- EN-TÊTE COLONNES (Âges depuis CALC!C) - DYNAMIQUES ---
ws_tcd['A2'] = "Genre"
ws_tcd['A2'].fill = header_fill
ws_tcd['A2'].font = header_font
ws_tcd['A2'].border = border
ws_tcd['A2'].alignment = center_align
ws_tcd.column_dimensions['A'].width = 15

# Insérer tous les âges en en-tête de colonne (CONDITIONNELLEMENT)
for col_idx in range(1, len_dict['len_age']):
    col_letter = get_column_letter(col_idx + 1)
    
    # N'affiche l'âge QUE si :
    # - Le filtre Âge = "Tous" (affiche tous les âges)
    # - OU l'âge de la colonne = le filtre Âge
    ws_tcd[f'{col_letter}2'] = (
        f'=IF(OR(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),)=TDB1!$D$3), '
        f'IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),""), "")'
    )
    ws_tcd[f'{col_letter}2'].fill = header_fill
    ws_tcd[f'{col_letter}2'].font = header_font
    ws_tcd[f'{col_letter}2'].border = border
    ws_tcd[f'{col_letter}2'].alignment = center_align
    ws_tcd.column_dimensions[col_letter].width = 12

# --- EN-TÊTE LIGNES (Genres depuis CALC!A) - DYNAMIQUES ---
for row_idx in range(1, len_dict['len_gender']):
    row_num = 2 + row_idx
    
    # N'affiche le genre QUE si :
    # - Le filtre Genre = "Tous" (affiche tous les genres)
    # - OU le genre de la ligne = le filtre Genre
    ws_tcd[f'A{row_num}'] = (
        f'=IF(OR(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),)=TDB1!$G$3), '
        f'IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),""), "")'
    )
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    ws_tcd[f'A{row_num}'].alignment = center_align

# --- DONNÉES : COUNTIFS avec affichage conditionnel ---
for row_idx in range(1, len_dict['len_gender']):
    row_num = 2 + row_idx
    for col_idx in range(1, len_dict['len_age']):
        col_letter = get_column_letter(col_idx + 1)
        
        # Affiche la valeur SEULEMENT si :
        # - L'âge de la colonne est affiché (correspond au filtre)
        # - ET le genre de la ligne est affiché (correspond au filtre)
        formula = (
            f'=IF(AND('
            f'OR(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),)=TDB1!$D$3), '
            f'OR(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),)=TDB1!$G$3)'
            f'), '
            f'IFERROR(COUNTIFS('
            f'DATA!$M:$M, 1, '
            f'DATA!$B:$B, IF(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),""), TDB1!$G$3), '
            f'DATA!$A:$A, IF(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),), TDB1!$D$3), '
            f'DATA!$D:$D, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3), '
            f'DATA!$I:$I, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
            f'), 0), "")'
        )
        
        ws_tcd[f'{col_letter}{row_num}'] = formula
        ws_tcd[f'{col_letter}{row_num}'].border = border
        ws_tcd[f'{col_letter}{row_num}'].alignment = center_align
        ws_tcd[f'{col_letter}{row_num}'].number_format = '0'

print("\n✅ Tableau TCD 'Dépression par Genre et Âge' créé (DYNAMIQUE) !")
print(f"   📊 Comportement :")
print(f"      ✓ Filtre Âge='Tous' + Genre='Tous' → Tableau complet (tous les genres × tous les âges)")
print(f"      ✓ Filtre Âge='19' + Genre='Tous' → 1 colonne (19) × tous les genres")
print(f"      ✓ Filtre Âge='Tous' + Genre='Female' → toutes les colonnes × 1 ligne (Female)")
print(f"      ✓ Filtre Âge='19' + Genre='Female' → 1 colonne (19) × 1 ligne (Female)\n")

wb.save(path_file)
wb.close()

print(f"📋 Feuilles présentes : {wb.sheetnames}\n")


✅ Tableau TCD 'Dépression par Genre et Âge' créé (DYNAMIQUE) !
   📊 Comportement :
      ✓ Filtre Âge='Tous' + Genre='Tous' → Tableau complet (tous les genres × tous les âges)
      ✓ Filtre Âge='19' + Genre='Tous' → 1 colonne (19) × tous les genres
      ✓ Filtre Âge='Tous' + Genre='Female' → toutes les colonnes × 1 ligne (Female)
      ✓ Filtre Âge='19' + Genre='Female' → 1 colonne (19) × 1 ligne (Female)

📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC', 'TCD']



### Tableau Addiction par genre et âge

In [680]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

wb = load_workbook(path_file)

# ========== STYLES ==========
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")  # Bleu foncé
title_font = Font(bold=True, size=12, color="FFFFFF")
header_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")  # Teal foncé
header_font = Font(color="FFFFFF", bold=True, size=10)
row_header_fill = PatternFill(start_color="FFB3E5E0", end_color="FFB3E5E0", fill_type="solid")  # Teal clair
row_header_font = Font(bold=True, color="000000", size=10)
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center")

# ========== CRÉATION DE LA FEUILLE ==========
if "TCD" in wb.sheetnames:
    ws_tcd = wb["TCD"]
else:
    ws_tcd = wb.create_sheet("TCD")

# =================================================================
# TABLEAU 2 : Addiction Moyenne par Genre et Âge (DYNAMIQUE)
# =================================================================

# --- TITRE (commencer à la ligne 10) ---
start_row = 10
ws_tcd[f'A{start_row}'] = "Addiction Moyenne par Genre et Âge"
ws_tcd[f'A{start_row}'].font = title_font
ws_tcd[f'A{start_row}'].fill = title_fill
ws_tcd[f'A{start_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{start_row}:J{start_row}')
ws_tcd.row_dimensions[start_row].height = 25

# --- EN-TÊTE COLONNES (Âges depuis CALC!C) - DYNAMIQUES ---
header_row = start_row + 1
ws_tcd[f'A{header_row}'] = "Genre"
ws_tcd[f'A{header_row}'].fill = header_fill
ws_tcd[f'A{header_row}'].font = header_font
ws_tcd[f'A{header_row}'].border = border
ws_tcd[f'A{header_row}'].alignment = center_align
ws_tcd.column_dimensions['A'].width = 15

# Insérer tous les âges en en-tête de colonne
for col_idx in range(1, len_dict['len_age']):
    col_letter = get_column_letter(col_idx + 1)
    
    # N'affiche l'âge QUE si :
    # - Le filtre Âge = "Tous" (affiche tous les âges)
    # - OU l'âge de la colonne = le filtre Âge
    ws_tcd[f'{col_letter}{header_row}'] = (
        f'=IF(OR(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),)=TDB1!$D$3), '
        f'IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),""), "")'
    )
    ws_tcd[f'{col_letter}{header_row}'].fill = header_fill
    ws_tcd[f'{col_letter}{header_row}'].font = header_font
    ws_tcd[f'{col_letter}{header_row}'].border = border
    ws_tcd[f'{col_letter}{header_row}'].alignment = center_align
    ws_tcd.column_dimensions[col_letter].width = 12

# --- EN-TÊTE LIGNES (Genres depuis CALC!A) - DYNAMIQUES ---
for row_idx in range(1, len_dict['len_gender']):
    row_num = header_row + row_idx
    
    # N'affiche le genre QUE si :
    # - Le filtre Genre = "Tous" (affiche tous les genres)
    # - OU le genre de la ligne = le filtre Genre
    ws_tcd[f'A{row_num}'] = (
        f'=IF(OR(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),)=TDB1!$G$3), '
        f'IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),""), "")'
    )
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    ws_tcd[f'A{row_num}'].alignment = center_align

# --- DONNÉES : AVERAGEIFS avec affichage conditionnel ---
for row_idx in range(1, len_dict['len_gender']):
    row_num = header_row + row_idx
    for col_idx in range(1, len_dict['len_age']):
        col_letter = get_column_letter(col_idx + 1)
        
        # Affiche la MOYENNE SEULEMENT si :
        # - L'âge de la colonne est affiché (correspond au filtre)
        # - ET le genre de la ligne est affiché (correspond au filtre)
        formula = (
            f'=IF(AND('
            f'OR(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),)=TDB1!$D$3), '
            f'OR(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),)=TDB1!$G$3)'
            f'), '
            f'IFERROR(AVERAGEIFS('
            f'DATA!$L:$L, '  # Colonne addiction_level
            f'DATA!$B:$B, IF(TDB1!$G$3="Tous", IFERROR(INDEX(CALC!$A$2:$A$100,{row_idx}),""), TDB1!$G$3), '
            f'DATA!$A:$A, IF(TDB1!$D$3="Tous", IFERROR(INDEX(CALC!$C$2:$C$100,{col_idx}),), TDB1!$D$3), '
            f'DATA!$D:$D, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3), '
            f'DATA!$I:$I, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
            f'), 0), "")'
        )
        
        ws_tcd[f'{col_letter}{row_num}'] = formula
        ws_tcd[f'{col_letter}{row_num}'].border = border
        ws_tcd[f'{col_letter}{row_num}'].alignment = center_align
        ws_tcd[f'{col_letter}{row_num}'].number_format = '0.00'  # Format décimal

print("\n✅ Tableau TCD 'Addiction Moyenne par Genre et Âge' créé (DYNAMIQUE) !")
print(f"   📊 Comportement :")
print(f"      ✓ Filtre Âge='Tous' + Genre='Tous' → Tableau complet (tous les genres × tous les âges)")
print(f"      ✓ Filtre Âge='19' + Genre='Tous' → 1 colonne (19) × tous les genres")
print(f"      ✓ Filtre Âge='Tous' + Genre='Female' → toutes les colonnes × 1 ligne (Female)")
print(f"      ✓ Filtre Âge='19' + Genre='Female' → 1 colonne (19) × 1 ligne (Female)")
print(f"   📈 Valeurs affichées : MOYENNE du niveau d'addiction\n")

wb.save(path_file)
wb.close()

print(f"📋 Feuilles présentes : {wb.sheetnames}\n")


✅ Tableau TCD 'Addiction Moyenne par Genre et Âge' créé (DYNAMIQUE) !
   📊 Comportement :
      ✓ Filtre Âge='Tous' + Genre='Tous' → Tableau complet (tous les genres × tous les âges)
      ✓ Filtre Âge='19' + Genre='Tous' → 1 colonne (19) × tous les genres
      ✓ Filtre Âge='Tous' + Genre='Female' → toutes les colonnes × 1 ligne (Female)
      ✓ Filtre Âge='19' + Genre='Female' → 1 colonne (19) × 1 ligne (Female)
   📈 Valeurs affichées : MOYENNE du niveau d'addiction

📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC', 'TCD']



### Summary pour boite à moustache

In [681]:
# from openpyxl import load_workbook
# from openpyxl.utils import get_column_letter
# from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
# from openpyxl.worksheet.formula import ArrayFormula

# wb = load_workbook(path_file)

# # ========== STYLES ==========
# title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")  # Bleu foncé
# title_font = Font(bold=True, size=12, color="FFFFFF")
# header_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")  # Teal foncé
# header_font = Font(color="FFFFFF", bold=True, size=10)
# row_header_fill = PatternFill(start_color="FFB3E5E0", end_color="FFB3E5E0", fill_type="solid")  # Teal clair
# row_header_font = Font(bold=True, color="000000", size=10)
# border = Border(
#     left=Side(style='thin'), right=Side(style='thin'),
#     top=Side(style='thin'), bottom=Side(style='thin')
# )
# center_align = Alignment(horizontal="center", vertical="center")

# # ========== CRÉATION DE LA FEUILLE ==========
# if "TCD" in wb.sheetnames:
#     ws_tcd = wb["TCD"]
# else:
#     ws_tcd = wb.create_sheet("TCD")

# # =================================================================
# # DÉFINITION DES FILTRES (Plages fixes et Matrices)
# # =================================================================
# # Positions des filtres en TDB1 :
# # - D3 = Âge
# # - G3 = Genre
# # - J3 = Plateforme
# # - M3 = Interaction

# filter_rng = (
#     ', DATA!$B$2:$B$1201, IF(TDB1!$G$3="Tous", "<>", TDB1!$G$3)'
#     ', DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)'
#     ', DATA!$D$2:$D$1201, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3)'
#     ', DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
# )

# # Filtres mathématiques pour les formules matricielles
# filter_arr = (
#     ' * IF(TDB1!$G$3="Tous", 1, DATA!$B$2:$B$1201=TDB1!$G$3)'
#     ' * IF(TDB1!$D$3="Tous", 1, DATA!$A$2:$A$1201=TDB1!$D$3)'
#     ' * IF(TDB1!$J$3="Tous", 1, DATA!$D$2:$D$1201=TDB1!$J$3)'
#     ' * IF(TDB1!$M$3="Tous", 1, DATA!$I$2:$I$1201=TDB1!$M$3)'
# )

# # =================================================================
# # TABLEAU 3 : Summary Statistiques Addiction par Âge
# # =================================================================

# start_row = 20

# ws_tcd[f'A{start_row}'] = "Statistiques Addiction par Âge (pour Boxplot)"
# ws_tcd[f'A{start_row}'].font = title_font
# ws_tcd[f'A{start_row}'].fill = title_fill
# ws_tcd[f'A{start_row}'].alignment = center_align
# ws_tcd.merge_cells(f'A{start_row}:H{start_row}')
# ws_tcd.row_dimensions[start_row].height = 25

# colonnes = ["Âge", "Min", "Q1", "Médiane", "Q3", "Max", "Moyenne", "Écart-type"]
# for col_idx, col_name in enumerate(colonnes):
#     col_letter = get_column_letter(col_idx + 1)
#     ws_tcd[f'{col_letter}{start_row+1}'] = col_name
#     ws_tcd[f'{col_letter}{start_row+1}'].fill = header_fill
#     ws_tcd[f'{col_letter}{start_row+1}'].font = header_font
#     ws_tcd[f'{col_letter}{start_row+1}'].border = border
#     ws_tcd[f'{col_letter}{start_row+1}'].alignment = center_align
#     ws_tcd.column_dimensions[col_letter].width = 12

# num_ages = len_dict['len_age']

# for row_idx in range(1, num_ages):
#     row_num = start_row + 1 + row_idx
    
#     # --- Colonne Âge ---
#     ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$C$2:$C$1000,{row_idx}),"")'
#     ws_tcd[f'A{row_num}'].fill = row_header_fill
#     ws_tcd[f'A{row_num}'].font = row_header_font
#     ws_tcd[f'A{row_num}'].border = border
#     ws_tcd[f'A{row_num}'].alignment = center_align
    
#     # --- Min ---
#     ws_tcd[f'B{row_num}'] = ArrayFormula(
#         f'B{row_num}', 
#         f'=MIN(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
#     )
    
#     # --- Q1 (25ème percentile) ---
#     ws_tcd[f'C{row_num}'] = ArrayFormula(
#         f'C{row_num}', 
#         f'=QUARTILE(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201), 1)'
#     )
    
#     # --- Médiane (50ème percentile) ---
#     ws_tcd[f'D{row_num}'] = ArrayFormula(
#         f'D{row_num}', 
#         f'=MEDIAN(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
#     )
    
#     # --- Q3 (75ème percentile) ---
#     ws_tcd[f'E{row_num}'] = ArrayFormula(
#         f'E{row_num}', 
#         f'=QUARTILE(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201), 3)'
#     )
    
#     # --- Max ---
#     ws_tcd[f'F{row_num}'] = ArrayFormula(
#         f'F{row_num}', 
#         f'=MAX(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
#     )
    
#     # --- Moyenne (utilise AVERAGEIFS standard) ---
#     ws_tcd[f'G{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$L$2:$L$1201, DATA!$A$2:$A$1201, $A{row_num}{filter_rng}), NA())'
    
#     # --- Écart-type ---
#     ws_tcd[f'H{row_num}'] = ArrayFormula(
#         f'H{row_num}', 
#         f'=STDEV(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
#     )
    
#     # Formatage
#     for col_idx in range(2, 9):
#         col_letter = get_column_letter(col_idx)
#         cell = ws_tcd[f'{col_letter}{row_num}']
#         cell.border = border
#         cell.alignment = center_align
#         cell.number_format = '0.00'

# # =================================================================
# # KPI SECTION
# # =================================================================

# kpi_start_row = start_row + num_ages + 3

# # --- TITRE SECTION KPI ---
# ws_tcd[f'A{kpi_start_row}'] = "Indicateurs Clés (KPI) - Personnes Dépressives"
# ws_tcd[f'A{kpi_start_row}'].font = Font(bold=True, size=12, color="FFFFFF")
# ws_tcd[f'A{kpi_start_row}'].fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
# ws_tcd[f'A{kpi_start_row}'].alignment = center_align
# ws_tcd.merge_cells(f'A{kpi_start_row}:H{kpi_start_row}')
# ws_tcd.row_dimensions[kpi_start_row].height = 25

# # ========== KPI 1 : RÉSEAU LE PLUS UTILISÉ ==========
# kpi1_row = kpi_start_row + 2

# ws_tcd[f'A{kpi1_row}'] = "Réseau le plus utilisé"
# ws_tcd[f'A{kpi1_row}'].fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
# ws_tcd[f'A{kpi1_row}'].font = Font(bold=True, color="FFFFFF", size=11)
# ws_tcd[f'A{kpi1_row}'].border = border
# ws_tcd[f'A{kpi1_row}'].alignment = center_align
# ws_tcd.merge_cells(f'A{kpi1_row}:C{kpi1_row}')

# kpi1_value_row = kpi1_row + 1
# ws_tcd[f'A{kpi1_value_row}'] = (
#     f'=INDEX(DATA!$D$2:$D$1201, MATCH(MAX(COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, '
#     f'DATA!$M$2:$M$1201, 1{filter_rng})), '
#     f'COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, DATA!$M$2:$M$1201, 1{filter_rng}), 0))'
# )
# ws_tcd[f'A{kpi1_value_row}'].fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
# ws_tcd[f'A{kpi1_value_row}'].font = Font(bold=True, size=14, color="FFFFFF")
# ws_tcd[f'A{kpi1_value_row}'].border = border
# ws_tcd[f'A{kpi1_value_row}'].alignment = center_align
# ws_tcd.merge_cells(f'A{kpi1_value_row}:C{kpi1_value_row}')
# ws_tcd.row_dimensions[kpi1_value_row].height = 30

# # Nombre de dépressifs
# kpi1_count_row = kpi1_value_row + 1
# ws_tcd[f'A{kpi1_count_row}'] = "Nombre de dépressifs"
# ws_tcd[f'A{kpi1_count_row}'].font = Font(bold=True, size=10)
# ws_tcd[f'A{kpi1_count_row}'].border = border

# ws_tcd[f'B{kpi1_count_row}'] = f'=IFERROR(COUNTIFS(DATA!$M$2:$M$1201, 1{filter_rng}), 0)'
# ws_tcd[f'B{kpi1_count_row}'].font = Font(bold=True, size=10)
# ws_tcd[f'B{kpi1_count_row}'].border = border
# ws_tcd[f'B{kpi1_count_row}'].alignment = center_align
# ws_tcd[f'B{kpi1_count_row}'].number_format = '0'

# # ========== KPI 2 : PERFORMANCE SCOLAIRE MOYENNE ==========
# kpi2_row = kpi_start_row + 2

# ws_tcd[f'E{kpi2_row}'] = "Performance scolaire moyenne"
# ws_tcd[f'E{kpi2_row}'].fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
# ws_tcd[f'E{kpi2_row}'].font = Font(bold=True, color="FFFFFF", size=11)
# ws_tcd[f'E{kpi2_row}'].border = border
# ws_tcd[f'E{kpi2_row}'].alignment = center_align
# ws_tcd.merge_cells(f'E{kpi2_row}:G{kpi2_row}')

# kpi2_value_row = kpi2_row + 1
# ws_tcd[f'E{kpi2_value_row}'] = (
#     f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, DATA!$M$2:$M$1201, 1{filter_rng}), NA())'
# )
# ws_tcd[f'E{kpi2_value_row}'].fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
# ws_tcd[f'E{kpi2_value_row}'].font = Font(bold=True, size=14, color="FFFFFF")
# ws_tcd[f'E{kpi2_value_row}'].border = border
# ws_tcd[f'E{kpi2_value_row}'].alignment = center_align
# ws_tcd.merge_cells(f'E{kpi2_value_row}:G{kpi2_value_row}')
# ws_tcd.row_dimensions[kpi2_value_row].height = 30
# ws_tcd[f'E{kpi2_value_row}'].number_format = '0.00'

# # Min / Max Performance
# kpi2_minmax_row = kpi2_value_row + 1
# ws_tcd[f'E{kpi2_minmax_row}'] = "Min / Max"
# ws_tcd[f'E{kpi2_minmax_row}'].font = Font(bold=True, size=10)
# ws_tcd[f'E{kpi2_minmax_row}'].border = border

# ws_tcd[f'F{kpi2_minmax_row}'] = (
#     f'=IFERROR(MINIFS(DATA!$G$2:$G$1201, DATA!$M$2:$M$1201, 1{filter_rng}), 0)'
# )
# ws_tcd[f'F{kpi2_minmax_row}'].font = Font(size=9)
# ws_tcd[f'F{kpi2_minmax_row}'].border = border
# ws_tcd[f'F{kpi2_minmax_row}'].alignment = center_align
# ws_tcd[f'F{kpi2_minmax_row}'].number_format = '0.00'

# ws_tcd[f'G{kpi2_minmax_row}'] = (
#     f'=IFERROR(MAXIFS(DATA!$G$2:$G$1201, DATA!$M$2:$M$1201, 1{filter_rng}), 0)'
# )
# ws_tcd[f'G{kpi2_minmax_row}'].font = Font(size=9)
# ws_tcd[f'G{kpi2_minmax_row}'].border = border
# ws_tcd[f'G{kpi2_minmax_row}'].alignment = center_align
# ws_tcd[f'G{kpi2_minmax_row}'].number_format = '0.00'

# print("\n✅ Tableau Summary et KPI créés avec succès !")
# print(f"   📊 Summary : Statistiques Addiction par Âge")
# print(f"      ✓ Min / Q1 / Médiane / Q3 / Max / Moyenne / Écart-type")

# wb.save(path_file)

# print(f"📋 Feuilles présentes : {wb.sheetnames}\n")

### KPI

In [682]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.worksheet.formula import ArrayFormula

wb = load_workbook(path_file)

# ========== STYLES ==========
title_fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
title_font = Font(bold=True, size=12, color="FFFFFF")
header_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True, size=10)
row_header_fill = PatternFill(start_color="FFB3E5E0", end_color="FFB3E5E0", fill_type="solid")
row_header_font = Font(bold=True, color="000000", size=10)
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center")

if "TCD" in wb.sheetnames:
    ws_tcd = wb["TCD"]
else:
    ws_tcd = wb.create_sheet("TCD")

# =================================================================
# DÉFINITION DES FILTRES
# =================================================================

# Filter complet : Genre + Âge + Plateforme + Interaction
filter_rng_complet = (
    ', DATA!$B$2:$B$1201, IF(TDB1!$G$3="Tous", "<>", TDB1!$G$3)'
    ', DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)'
    ', DATA!$D$2:$D$1201, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3)'
    ', DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
)

# Filter SANS Plateforme (pour trouver le réseau le plus utilisé)
filter_rng_sans_plateforme = (
    ', DATA!$B$2:$B$1201, IF(TDB1!$G$3="Tous", "<>", TDB1!$G$3)'
    ', DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)'
    ', DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
)

# Filtres matriciels pour ArrayFormula
filter_arr = (
    ' * IF(TDB1!$G$3="Tous", 1, DATA!$B$2:$B$1201=TDB1!$G$3)'
    ' * IF(TDB1!$D$3="Tous", 1, DATA!$A$2:$A$1201=TDB1!$D$3)'
    ' * IF(TDB1!$J$3="Tous", 1, DATA!$D$2:$D$1201=TDB1!$J$3)'
    ' * IF(TDB1!$M$3="Tous", 1, DATA!$I$2:$I$1201=TDB1!$M$3)'
)

# Filtres matriciels SANS Plateforme
filter_arr_sans_plateforme = (
    ' * IF(TDB1!$G$3="Tous", 1, DATA!$B$2:$B$1201=TDB1!$G$3)'
    ' * IF(TDB1!$D$3="Tous", 1, DATA!$A$2:$A$1201=TDB1!$D$3)'
    ' * IF(TDB1!$M$3="Tous", 1, DATA!$I$2:$I$1201=TDB1!$M$3)'
)

# =================================================================
# TABLEAU 3 : Summary Statistiques Addiction par Âge
# =================================================================

start_row = 20

ws_tcd[f'A{start_row}'] = "Statistiques Addiction par Âge (pour Boxplot)"
ws_tcd[f'A{start_row}'].font = title_font
ws_tcd[f'A{start_row}'].fill = title_fill
ws_tcd[f'A{start_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{start_row}:H{start_row}')
ws_tcd.row_dimensions[start_row].height = 25

colonnes = ["Âge", "Min", "Q1", "Médiane", "Q3", "Max", "Moyenne", "Écart-type"]
for col_idx, col_name in enumerate(colonnes):
    col_letter = get_column_letter(col_idx + 1)
    ws_tcd[f'{col_letter}{start_row+1}'] = col_name
    ws_tcd[f'{col_letter}{start_row+1}'].fill = header_fill
    ws_tcd[f'{col_letter}{start_row+1}'].font = header_font
    ws_tcd[f'{col_letter}{start_row+1}'].border = border
    ws_tcd[f'{col_letter}{start_row+1}'].alignment = center_align
    ws_tcd.column_dimensions[col_letter].width = 12

num_ages = len_dict['len_age']

for row_idx in range(1, num_ages):
    row_num = start_row + 1 + row_idx
    
    ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$C$2:$C$1000,{row_idx}),"")'
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    ws_tcd[f'A{row_num}'].alignment = center_align
    
    ws_tcd[f'B{row_num}'] = ArrayFormula(
        f'B{row_num}', 
        f'=MIN(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
    )
    
    ws_tcd[f'C{row_num}'] = ArrayFormula(
        f'C{row_num}', 
        f'=QUARTILE(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201), 1)'
    )
    
    ws_tcd[f'D{row_num}'] = ArrayFormula(
        f'D{row_num}', 
        f'=MEDIAN(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
    )
    
    ws_tcd[f'E{row_num}'] = ArrayFormula(
        f'E{row_num}', 
        f'=QUARTILE(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201), 3)'
    )
    
    ws_tcd[f'F{row_num}'] = ArrayFormula(
        f'F{row_num}', 
        f'=MAX(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
    )
    
    ws_tcd[f'G{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$L$2:$L$1201, DATA!$A$2:$A$1201, $A{row_num}, DATA!$M$2:$M$1201, 1{filter_rng_complet}), NA())'
    
    ws_tcd[f'H{row_num}'] = ArrayFormula(
        f'H{row_num}', 
        f'=STDEV(IF((DATA!$A$2:$A$1201=$A{row_num}){filter_arr}, DATA!$L$2:$L$1201))'
    )
    
    for col_idx in range(2, 9):
        col_letter = get_column_letter(col_idx)
        cell = ws_tcd[f'{col_letter}{row_num}']
        cell.border = border
        cell.alignment = center_align
        cell.number_format = '0.00'



wb.save(path_file)
wb.close()

print(f"📋 Feuilles présentes : {wb.sheetnames}\n")

📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC', 'TCD']



In [683]:
# from openpyxl import load_workbook
# from openpyxl.worksheet.formula import ArrayFormula

# wb = load_workbook(path_file)

# if "TCD" in wb.sheetnames:
#     ws_tcd = wb["TCD"]
# else:
#     ws_tcd = wb.create_sheet("TCD")

# # =================================================================
# # DÉFINITION DES FILTRES
# # =================================================================

# filter_rng = (
#     ', DATA!$B$2:$B$1201, IF(TDB1!$G$3="Tous", "<>", TDB1!$G$3)'
#     ', DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)'
#     ', DATA!$D$2:$D$1201, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3)'
#     ', DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
# )

# # =================================================================
# # KPI 1 : RÉSEAU LE PLUS UTILISÉ
# # =================================================================

# kpi_start_row = 30

# ws_tcd[f"A{kpi_start_row}"] = "Réseau le plus utilisé"

# # FORMULE ADAPTÉE : Compter les dépressifs (M=1) par plateforme avec filtres
# formule_plateforme = (
#     f'=INDEX(DATA!$D$2:$D$1201, '
#     f'MATCH(MAX(COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, '
#     f'DATA!$M$2:$M$1201, 1{filter_rng})), '
#     f'COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, '
#     f'DATA!$M$2:$M$1201, 1{filter_rng}), 0))'
# )

# ws_tcd[f"B{kpi_start_row}"] = ArrayFormula(f"B{kpi_start_row}", formule_plateforme)

# # =================================================================
# # KPI 2 : PERFORMANCE SCOLAIRE MOYENNE
# # =================================================================

# ws_tcd[f"A{kpi_start_row+1}"] = "Perf sco moyenne - dépressifs"

# formule_perf = (
#     f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, '
#     f'DATA!$M$2:$M$1201, 1{filter_rng}), NA())'
# )

# ws_tcd[f"B{kpi_start_row+1}"] = formule_perf

# print("\n✅ KPI corrigés avec la formule adaptée !")
# print(f"   📊 KPI 1 : Réseau le plus utilisé")
# print(f"      ✓ Compte chaque plateforme chez les dépressifs")
# print(f"   📊 KPI 2 : Performance scolaire moyenne")
# print(f"   🔗 Filtres appliqués : Genre (G3) + Âge (D3) + Plateforme (J3) + Interaction (M3)\n")

# wb.save(path_file)
# wb.close()

# print(f"📋 Feuilles présentes : {wb.sheetnames}\n")

In [684]:
from openpyxl import load_workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

wb = load_workbook(path_file)

if "TCD" in wb.sheetnames:
    ws_tcd = wb["TCD"]
else:
    ws_tcd = wb.create_sheet("TCD")

# ========== STYLES ==========
kpi_title_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
kpi_title_font = Font(bold=True, color="FFFFFF", size=11)
kpi_value_fill = PatternFill(start_color="FF008080", end_color="FF008080", fill_type="solid")
kpi_value_font = Font(bold=True, size=16, color="FFFFFF")
kpi_label_font = Font(bold=True, size=10, color="000000")
border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

# =================================================================
# DÉFINITION DES FILTRES
# =================================================================

filter_rng = (
    ', DATA!$B$2:$B$1201, IF(TDB1!$G$3="Tous", "<>", TDB1!$G$3)'
    ', DATA!$A$2:$A$1201, IF(TDB1!$D$3="Tous", "<>", TDB1!$D$3)'
    ', DATA!$D$2:$D$1201, IF(TDB1!$J$3="Tous", "<>", TDB1!$J$3)'
    ', DATA!$I$2:$I$1201, IF(TDB1!$M$3="Tous", "<>", TDB1!$M$3)'
)

# Filtres matriciels pour ArrayFormula
filter_arr = (
    ' * IF(TDB1!$G$3="Tous", 1, DATA!$B$2:$B$1201=TDB1!$G$3)'
    ' * IF(TDB1!$D$3="Tous", 1, DATA!$A$2:$A$1201=TDB1!$D$3)'
    ' * IF(TDB1!$J$3="Tous", 1, DATA!$D$2:$D$1201=TDB1!$J$3)'
    ' * IF(TDB1!$M$3="Tous", 1, DATA!$I$2:$I$1201=TDB1!$M$3)'
)

# =================================================================
# KPI SECTION
# =================================================================

kpi_start_row = 30

# --- TITRE SECTION KPI ---
ws_tcd[f'A{kpi_start_row}'] = "Indicateurs Clés (KPI) - Personnes Dépressives"
ws_tcd[f'A{kpi_start_row}'].font = Font(bold=True, size=12, color="FFFFFF")
ws_tcd[f'A{kpi_start_row}'].fill = PatternFill(start_color="FF003D6B", end_color="FF003D6B", fill_type="solid")
ws_tcd[f'A{kpi_start_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{kpi_start_row}:H{kpi_start_row}')
ws_tcd.row_dimensions[kpi_start_row].height = 25

# ========== KPI 1 : RÉSEAU LE PLUS UTILISÉ ==========
kpi1_row = kpi_start_row + 2

ws_tcd[f'A{kpi1_row}'] = "Réseau le plus utilisé"
ws_tcd[f'A{kpi1_row}'].fill = kpi_title_fill
ws_tcd[f'A{kpi1_row}'].font = kpi_title_font
ws_tcd[f'A{kpi1_row}'].border = border
ws_tcd[f'A{kpi1_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{kpi1_row}:C{kpi1_row}')
ws_tcd.row_dimensions[kpi1_row].height = 20

kpi1_value_row = kpi1_row + 1
formule_plateforme = (
    f'=INDEX(DATA!$D$2:$D$1201, '
    f'MATCH(MAX(COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, '
    f'DATA!$M$2:$M$1201, 1{filter_rng})), '
    f'COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, '
    f'DATA!$M$2:$M$1201, 1{filter_rng}), 0))'
)
ws_tcd[f'A{kpi1_value_row}'] = ArrayFormula(f"A{kpi1_value_row}", formule_plateforme)
ws_tcd[f'A{kpi1_value_row}'].fill = kpi_value_fill
ws_tcd[f'A{kpi1_value_row}'].font = kpi_value_font
ws_tcd[f'A{kpi1_value_row}'].border = border
ws_tcd[f'A{kpi1_value_row}'].alignment = center_align
ws_tcd.merge_cells(f'A{kpi1_value_row}:C{kpi1_value_row}')
ws_tcd.row_dimensions[kpi1_value_row].height = 35

# Nombre de dépressifs
kpi1_count_row = kpi1_value_row + 1
ws_tcd[f'A{kpi1_count_row}'] = "Nombre de dépressifs"
ws_tcd[f'A{kpi1_count_row}'].font = kpi_label_font
ws_tcd[f'A{kpi1_count_row}'].border = border
ws_tcd[f'A{kpi1_count_row}'].alignment = Alignment(horizontal="left", vertical="center")

ws_tcd[f'B{kpi1_count_row}'] = (
    f'=IFERROR(COUNTIFS(DATA!$M$2:$M$1201, 1{filter_rng}), 0)'
)
ws_tcd[f'B{kpi1_count_row}'].font = Font(bold=True, size=11)
ws_tcd[f'B{kpi1_count_row}'].border = border
ws_tcd[f'B{kpi1_count_row}'].alignment = center_align
ws_tcd[f'B{kpi1_count_row}'].number_format = '0'

ws_tcd.column_dimensions['A'].width = 18
ws_tcd.column_dimensions['B'].width = 12
ws_tcd.column_dimensions['C'].width = 12

# ========== KPI 2 : PERFORMANCE SCOLAIRE MOYENNE ==========
kpi2_row = kpi_start_row + 2

ws_tcd[f'D{kpi2_row}'] = "Performance scolaire moyenne"
ws_tcd[f'D{kpi2_row}'].fill = kpi_title_fill
ws_tcd[f'D{kpi2_row}'].font = kpi_title_font
ws_tcd[f'D{kpi2_row}'].border = border
ws_tcd[f'D{kpi2_row}'].alignment = center_align
ws_tcd.merge_cells(f'D{kpi2_row}:F{kpi2_row}')
ws_tcd.row_dimensions[kpi2_row].height = 20

kpi2_value_row = kpi2_row + 1
ws_tcd[f'D{kpi2_value_row}'] = (
    f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, '
    f'DATA!$M$2:$M$1201, 1{filter_rng}), NA())'
)
ws_tcd[f'D{kpi2_value_row}'].fill = kpi_value_fill
ws_tcd[f'D{kpi2_value_row}'].font = kpi_value_font
ws_tcd[f'D{kpi2_value_row}'].border = border
ws_tcd[f'D{kpi2_value_row}'].alignment = center_align
ws_tcd.merge_cells(f'D{kpi2_value_row}:F{kpi2_value_row}')
ws_tcd.row_dimensions[kpi2_value_row].height = 35
ws_tcd[f'D{kpi2_value_row}'].number_format = '0.00'

# Min / Max Performance - CORRECTION : Utiliser ArrayFormula
kpi2_minmax_row = kpi2_value_row + 1
ws_tcd[f'D{kpi2_minmax_row}'] = "Min / Max"
ws_tcd[f'D{kpi2_minmax_row}'].font = kpi_label_font
ws_tcd[f'D{kpi2_minmax_row}'].border = border
ws_tcd[f'D{kpi2_minmax_row}'].alignment = Alignment(horizontal="left", vertical="center")

# MIN avec ArrayFormula
ws_tcd[f'E{kpi2_minmax_row}'] = ArrayFormula(
    f'E{kpi2_minmax_row}',
    f'=MIN(IF((DATA!$M$2:$M$1201=1){filter_arr}, DATA!$G$2:$G$1201))'
)
ws_tcd[f'E{kpi2_minmax_row}'].font = Font(size=9, bold=True)
ws_tcd[f'E{kpi2_minmax_row}'].border = border
ws_tcd[f'E{kpi2_minmax_row}'].alignment = center_align
ws_tcd[f'E{kpi2_minmax_row}'].number_format = '0.00'

# MAX avec ArrayFormula
ws_tcd[f'F{kpi2_minmax_row}'] = ArrayFormula(
    f'F{kpi2_minmax_row}',
    f'=MAX(IF((DATA!$M$2:$M$1201=1){filter_arr}, DATA!$G$2:$G$1201))'
)
ws_tcd[f'F{kpi2_minmax_row}'].font = Font(size=9, bold=True)
ws_tcd[f'F{kpi2_minmax_row}'].border = border
ws_tcd[f'F{kpi2_minmax_row}'].alignment = center_align
ws_tcd[f'F{kpi2_minmax_row}'].number_format = '0.00'

ws_tcd.column_dimensions['D'].width = 18
ws_tcd.column_dimensions['E'].width = 12
ws_tcd.column_dimensions['F'].width = 12

print("\n✅ KPI avec Min/Max corrigés (ArrayFormula) !")
print(f"   📊 KPI 1 - Réseau le plus utilisé")
print(f"      ✓ Plateforme la plus utilisée")
print(f"      ✓ Nombre total de dépressifs")
print(f"   📊 KPI 2 - Performance scolaire")
print(f"      ✓ Moyenne")
print(f"      ✓ Min / Max (avec ArrayFormula)\n")

wb.save(path_file)
wb.close()

print(f"📋 Feuilles présentes : {wb.sheetnames}\n")


✅ KPI avec Min/Max corrigés (ArrayFormula) !
   📊 KPI 1 - Réseau le plus utilisé
      ✓ Plateforme la plus utilisée
      ✓ Nombre total de dépressifs
   📊 KPI 2 - Performance scolaire
      ✓ Moyenne
      ✓ Min / Max (avec ArrayFormula)

📋 Feuilles présentes : ['TDB1', 'DATA', 'Correlations', 'CALC', 'TCD']



# Création des graphiques

### BarChart - Niveau d'addiction moyen selon l'âge et le genre

In [689]:
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.drawing.text import RichTextProperties, Paragraph, ParagraphProperties, CharacterProperties
from openpyxl.styles.colors import Color

wb = load_workbook(path_file)

ws_tcd = wb["TCD"]
ws_tdb1 = wb["TDB1"]
# =================================================================
# GRAPHIQUE 1 : Addiction Moyenne par Âge et Genre (CORRIGÉ)
# =================================================================

chart_addiction = BarChart()
chart_addiction.type = "col"
chart_addiction.style = 10
chart_addiction.title = "Addiction moyenne par âge et genre" # Titre simple
chart_addiction.y_axis.title = "Niveau d'addiction"
chart_addiction.x_axis.title = "Âge"
chart_addiction.height = 10
chart_addiction.width = 10

# Références de données
max_col = len_dict['len_age']
data = Reference(ws_tcd, min_col=1, min_row=12, max_col=max_col, max_row=13)
cats = Reference(ws_tcd, min_col=2, min_row=11, max_col=max_col, max_row=11)

chart_addiction.add_data(data, titles_from_data=True, from_rows=True)
chart_addiction.set_categories(cats)
chart_addiction.overlap = -15

# Configuration des étiquettes de données (DataLabels)
for series in chart_addiction.series:
    series.dLbls = DataLabelList()
    series.dLbls.showVal = True       # Affiche la valeur
    series.dLbls.showCatName = False
    series.dLbls.showSerName = False
    series.dLbls.position = 'outEnd'  # Position externe
    series.dLbls.numFmt = "0.0"       # Format numérique

# Couleurs définies
color_male = "4169E1"
color_female = "FF1493"

# Configuration des séries avec couleurs pour barres ET étiquettes
if len(chart_addiction.series) >= 2:
    # --- Série 0 : Male (Bleu) ---
    chart_addiction.series[0].graphicalProperties.solidFill = color_male
    for point in chart_addiction.series[0].data_points:
        point.graphicalProperties.solidFill = color_male
    
    # --- Série 1 : Female (Rose) ---
    chart_addiction.series[1].graphicalProperties.solidFill = color_female
    for point in chart_addiction.series[1].data_points:
        point.graphicalProperties.solidFill = color_female

    for i, series in enumerate(chart_addiction.series):
        series.dLbls = DataLabelList()
        series.dLbls.showVal = True
        series.dLbls.position = 'outEnd'
        series.dLbls.numFmt = "0.0"
        
        # Définition de la couleur (hexadécimal sans le #)
        hex_color = color_male if i == 0 else color_female
        
        # Création de la couleur avec Color(srgbClr=...)
        # srgbClr attend la valeur hexadécimale
        text_color = Color(srgbClr=hex_color)
        
        # Application via CharacterProperties
        series.dLbls.txPr = RichTextProperties(
            p=[Paragraph(
                pPr=ParagraphProperties(
                    defRPr=CharacterProperties(solidFill=text_color)
                )
            )]
        )

chart_addiction.legend.position = "r"
chart_addiction.y_axis.majorGridlines = None

ws_tdb1.add_chart(chart_addiction, "B6")

wb.save(path_file)
wb.close()

print(f"✅ Fichier sauvegardé !\n")

TypeError: Color.__init__() got an unexpected keyword argument 'srgbClr'

### PieChart - Répartition de la dépression selon le genre

### Boîte à moustache - Répartition niveau addiction selon l'âge